# Real-World Accuracy Enhancement Pipeline
This pipeline focuses on extracting real-world accuracy through Best Practices: Feature Engineering, Cross-Validation, and XGBoost Hyperparameter Tuning.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), 'src')))
from data_processing import clean_data

### 1. Data Loading & Preprocessing

In [2]:
df = pd.read_csv('Student Mental health.csv')

# Impute age median to avoid NaN drops
df['Age'] = df['Age'].fillna(df['Age'].median())

df = clean_data(df)
df.head()

### 2. Feature Engineering
We introduce `Symptom_Severity` to help the model quantify combined psychological strain before predicting depression.

In [3]:
X = df.drop('Do you have Depression?', axis=1)
y = df['Do you have Depression?']

X['Symptom_Severity'] = X['Do you have Anxiety?'] + X['Do you have Panic attack?']
X.head()

### 3. Scaling & Train-Test Split

In [4]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

### 4. Advanced Modeling (Random Forest) with GridSearchCV
Instead of blind guessing, we use Grid Search to cross-validate multiple hyperparameter combinations for an Optimized Random Forest.

In [5]:
rf_model = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print("Optimal Settings Found:", grid_search.best_params_)

### 5. Evaluation & Export

In [6]:
y_pred = best_model.predict(X_test)
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [7]:
pickle.dump(best_model, open("mental_health_model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))